# Problem E — Contributing Factor Graph: Visualizations

Replicates and extends all charts from the Streamlit **Problem E** page.

| Section | Charts |
|---------|--------|
| Network | Interactive spring-layout graph (color by type / community / severity) |
| Centrality | Betweenness + weighted-degree bar charts, severity vs betweenness scatter |
| Patterns | Top factor patterns bar, severity vs lift scatter |
| Extra EDA | Community composition, node-type severity boxplots |

**Pre-req:** Run `scripts/run_phase5.py` to generate `data/processed/factor_graph.json`.

In [ ]:
import os
from pathlib import Path

root = Path.cwd()
while not (root / "pyproject.toml").exists():
    root = root.parent
os.chdir(root)
print(f"Working directory: {root}")

In [ ]:
import json
import numpy as np
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px

PROCESSED = Path("data/processed")

NODE_TYPE_COLORS = {
    "FACTOR":   "#e74c3c",
    "PHASE":    "#3498db",
    "AIRCRAFT": "#2ecc71",
    "FAR":      "#f39c12",
}
COMMUNITY_PALETTE = px.colors.qualitative.Plotly

print("Libraries loaded.")

## 1. Load Graph Data

In [ ]:
with open(PROCESSED / "factor_graph.json") as f:
    graph_data = json.load(f)

G          = nx.node_link_graph(graph_data, edges="links")
centrality = pd.read_csv(PROCESSED / "centrality_report.csv")

patterns_path = PROCESSED / "factor_patterns.json"
patterns = json.loads(patterns_path.read_text()) if patterns_path.exists() else []

n_communities = len(set(nx.get_node_attributes(G, "community").values()))

print(f"Nodes:       {G.number_of_nodes():,}")
print(f"Edges:       {G.number_of_edges():,}")
print(f"Communities: {n_communities}")
print(f"Patterns:    {len(patterns)}")
print(f"\nNode types: {pd.Series(nx.get_node_attributes(G, 'node_type')).value_counts().to_dict()}")

## 2. Network Graph — Colored by Node Type

Adjust `min_weight` to reduce clutter.

In [ ]:
min_weight = 200  # minimum co-occurrence count to show edge
color_by   = "Node Type"  # "Node Type" | "Community" | "Avg Severity"
show_labels = True

# Filter to subgraph
nodes_in = set()
for u, v, d in G.edges(data=True):
    if d.get("weight", 0) >= min_weight:
        nodes_in.add(u)
        nodes_in.add(v)

H   = G.subgraph(nodes_in)
pos = nx.spring_layout(H, seed=42, k=1.5 / max(1, H.number_of_nodes() ** 0.5))

# Edges
ex, ey = [], []
for u, v in H.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    ex += [x0, x1, None]
    ey += [y0, y1, None]

edge_trace = go.Scatter(x=ex, y=ey, mode="lines",
                        line=dict(width=0.5, color="#cccccc"),
                        hoverinfo="none", showlegend=False)

# Nodes (one trace per type for the legend)
node_types  = sorted(set(nx.get_node_attributes(H, "node_type").values()))
node_traces = []

for nt in node_types:
    nodes_nt = [n for n, d in H.nodes(data=True) if d.get("node_type") == nt]
    if not nodes_nt:
        continue

    xs, ys, texts, hovers, sizes, colors_arr = [], [], [], [], [], []
    for n in nodes_nt:
        x, y       = pos[n]
        d          = H.nodes[n]
        count      = d.get("count", 1)
        avg_sev    = d.get("avg_severity", 0)
        community  = d.get("community", 0)
        short_name = n.split(":", 1)[1] if ":" in n else n

        xs.append(x); ys.append(y)
        texts.append(short_name if show_labels else "")
        hovers.append(
            f"<b>{short_name}</b><br>Type: {nt}<br>"
            f"Reports: {count:,}<br>Avg Severity: {avg_sev:.3f}<br>Community: {community}"
        )
        sizes.append(max(8, min(35, np.log1p(count) * 3)))

        if color_by == "Community":
            colors_arr.append(COMMUNITY_PALETTE[community % len(COMMUNITY_PALETTE)])
        elif color_by == "Avg Severity":
            colors_arr.append(avg_sev)
        else:
            colors_arr.append(NODE_TYPE_COLORS.get(nt, "#999"))

    node_traces.append(go.Scatter(
        x=xs, y=ys,
        mode="markers+text" if show_labels else "markers",
        text=texts, textposition="top center", textfont=dict(size=9),
        marker=dict(size=sizes, color=colors_arr, line=dict(width=1, color="white")),
        hovertext=hovers, hoverinfo="text",
        name=nt,
    ))

fig = go.Figure(data=[edge_trace] + node_traces)
fig.update_layout(
    title=f"Factor Co-occurrence Graph (edges ≥ {min_weight} co-occurrences)",
    showlegend=True,
    hovermode="closest",
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=650,
    margin=dict(t=60, b=10, l=10, r=10),
    legend=dict(title="Node Type", orientation="v", yanchor="top", y=1, xanchor="right", x=1),
)
fig.show()

## 3. Betweenness Centrality — Top Nodes

In [ ]:
top_n = 20
centrality["short_name"] = centrality["node"].str.split(":").str[-1]

df_bc = centrality.sort_values("betweenness", ascending=False).head(top_n)

fig = px.bar(
    df_bc,
    x="betweenness",
    y="short_name",
    orientation="h",
    color="type",
    color_discrete_map=NODE_TYPE_COLORS,
    hover_data={"count": True, "avg_severity": ":.3f"},
    title=f"Top {top_n} Nodes by Betweenness Centrality",
    labels={"betweenness": "Betweenness", "short_name": "Node"},
)
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=max(400, top_n * 24), margin=dict(l=200))
fig.show()

## 4. Weighted Degree — Top Nodes

In [ ]:
df_wd = centrality.sort_values("weighted_degree", ascending=False).head(top_n)

fig = px.bar(
    df_wd,
    x="weighted_degree",
    y="short_name",
    orientation="h",
    color="type",
    color_discrete_map=NODE_TYPE_COLORS,
    hover_data={"count": True, "avg_severity": ":.3f"},
    title=f"Top {top_n} Nodes by Weighted Degree (co-occurrence volume)",
    labels={"weighted_degree": "Weighted Degree", "short_name": "Node"},
)
fig.update_yaxes(autorange="reversed")
fig.update_layout(height=max(400, top_n * 24), margin=dict(l=200))
fig.show()

## 5. Severity vs Betweenness Centrality

In [ ]:
fig = px.scatter(
    centrality,
    x="betweenness",
    y="avg_severity",
    size="count",
    color="type",
    color_discrete_map=NODE_TYPE_COLORS,
    hover_name="short_name",
    hover_data={"count": True, "weighted_degree": True},
    title="Node Avg Severity vs. Betweenness Centrality (bubble = report count)",
    labels={"betweenness": "Betweenness", "avg_severity": "Avg Severity"},
)
fig.add_hline(
    y=centrality["avg_severity"].mean(),
    line_dash="dash", line_color="gray",
    annotation_text="Mean severity",
)
fig.update_layout(height=460)
fig.show()

## 6. Node-Type Severity Distribution (Boxplot)

In [ ]:
fig = go.Figure()
for nt, color in NODE_TYPE_COLORS.items():
    sub = centrality[centrality["type"] == nt]["avg_severity"]
    if sub.empty:
        continue
    fig.add_trace(go.Box(
        y=sub, name=nt, marker_color=color, boxmean=True,
    ))
fig.update_layout(
    title="Avg Severity Distribution by Node Type",
    yaxis_title="Avg Severity",
    height=400,
)
fig.show()

## 7. Community Composition

In [ ]:
node_attrs = pd.DataFrame.from_dict(
    dict(G.nodes(data=True)), orient="index"
)

if "community" in node_attrs.columns and "node_type" in node_attrs.columns:
    comm_comp = (
        node_attrs.groupby(["community", "node_type"])
        .size()
        .reset_index(name="count")
    )

    fig = px.bar(
        comm_comp,
        x="community",
        y="count",
        color="node_type",
        color_discrete_map=NODE_TYPE_COLORS,
        barmode="stack",
        title="Community Composition by Node Type",
        labels={"community": "Community ID", "count": "Node Count", "node_type": "Type"},
    )
    fig.update_layout(height=420)
    fig.show()
else:
    print("community or node_type attributes not found in graph nodes.")

## 8. Factor Patterns — Top Patterns by Risk Score

In [ ]:
if not patterns:
    print("No patterns found — run scripts/run_phase5.py first.")
else:
    df_pat = pd.DataFrame(patterns)
    df_pat["factors_str"] = df_pat["factors"].apply(
        lambda f: " + ".join(f) if isinstance(f, list) else str(f)
    )
    df_pat["risk_proxy"]  = df_pat["avg_severity"] * df_pat["lift"]
    df_pat_sorted         = df_pat.sort_values("risk_proxy", ascending=False).reset_index(drop=True)

    fig = px.bar(
        df_pat_sorted.head(15),
        x="risk_proxy",
        y="factors_str",
        orientation="h",
        color="avg_severity",
        color_continuous_scale="RdYlGn_r",
        hover_data={"support": True, "lift": ":.2f", "avg_severity": ":.3f"},
        title="Top 15 Factor Patterns (sorted by Severity × Lift)",
        labels={"risk_proxy": "Severity × Lift", "factors_str": "Pattern", "avg_severity": "Avg Severity"},
    )
    fig.update_yaxes(autorange="reversed")
    fig.update_layout(height=460, margin=dict(l=300))
    fig.show()

## 9. Factor Patterns — Severity vs Lift Scatter

In [ ]:
if patterns:
    fig = px.scatter(
        df_pat_sorted,
        x="lift",
        y="avg_severity",
        size="support",
        color="risk_proxy",
        color_continuous_scale="RdYlGn_r",
        hover_name="factors_str",
        hover_data={"support": True},
        title="Factor Patterns: Severity vs Lift (bubble = incident support)",
        labels={"lift": "Lift", "avg_severity": "Avg Severity", "risk_proxy": "Sev × Lift"},
    )
    fig.add_hline(y=1.5, line_dash="dash", line_color="#e74c3c", annotation_text="High-severity threshold")
    fig.add_vline(x=1.5, line_dash="dash", line_color="#e74c3c", annotation_text="High-lift threshold")
    fig.add_hline(y=1.0, line_dash="dot",  line_color="gray", annotation_text="Severity = 1")
    fig.add_vline(x=1.0, line_dash="dot",  line_color="gray", annotation_text="Lift = 1")
    fig.update_layout(height=460)
    fig.show()

    # Highlight high-risk patterns
    high_risk = df_pat_sorted[
        (df_pat_sorted["lift"] > 1.5) & (df_pat_sorted["avg_severity"] > 1.5)
    ]
    print(f"\nHigh-risk patterns (lift > 1.5 AND avg_severity > 1.5): {len(high_risk)}")
    if not high_risk.empty:
        print(high_risk[["factors_str", "support", "avg_severity", "lift", "risk_proxy"]]
              .rename(columns={"factors_str": "Factors", "risk_proxy": "Sev×Lift"})
              .to_string(index=False))

## 10. Full Centrality Table

In [ ]:
display_cent = centrality[[
    "short_name", "type", "count", "avg_severity",
    "degree_centrality", "betweenness", "weighted_degree"
]].copy()
display_cent.columns = ["Node", "Type", "Reports", "Avg Severity",
                        "Degree Centrality", "Betweenness", "Weighted Degree"]
for col in ["Avg Severity", "Degree Centrality", "Betweenness"]:
    display_cent[col] = display_cent[col].round(4)

display_cent = display_cent.sort_values("Betweenness", ascending=False)
print(display_cent.set_index("Node").head(30).to_string())